# Training a SAE on T5-large Decoder with SAELens

This notebook trains a Sparse Autoencoder (SAE) on the **decoder** side of T5-large using SAELens's `StandardTrainingSAE`.

Since SAELens's `SAETrainingRunner` does not natively support `HookedEncoderDecoder`, we:
1. Load T5-large via TransformerLens's `HookedEncoderDecoder`
2. Manually collect activations from a decoder hook point
3. Train `StandardTrainingSAE` with a manual training loop using `TrainStepInput` / `training_forward_pass`

We use the XSum summarization dataset to provide meaningful encoder-decoder pairs.

## Setup

In [ ]:
try:
    import google.colab  # noqa: F401
    %pip install sae-lens transformer-lens datasets
except ImportError:
    from IPython import get_ipython
    ipython = get_ipython()
    if ipython is not None:
        ipython.run_line_magic("load_ext", "autoreload")
        ipython.run_line_magic("autoreload", "2")

In [ ]:
import torch
import os
from transformer_lens import HookedEncoderDecoder
from sae_lens.saes.standard_sae import StandardTrainingSAE, StandardTrainingSAEConfig
from sae_lens.saes.sae import TrainStepInput

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Load T5-large via TransformerLens

T5 is an encoder-decoder model. We use `HookedEncoderDecoder` to load it.
The decoder has 24 layers, each with hook points like `decoder.{i}.hook_mlp_out`.

In [ ]:
model = HookedEncoderDecoder.from_pretrained("google-t5/t5-large")
model.eval()

print(f"Model: google-t5/t5-large")
print(f"  d_model = {model.cfg.d_model}")
print(f"  d_mlp   = {model.cfg.d_mlp}")
print(f"  n_heads = {model.cfg.n_heads}")
print(f"  n_layers = {model.cfg.n_layers}")

## Test: Run T5 with encoder input + decoder input

T5 requires both encoder input (source text) and decoder input (target text prefix).
We use `run_with_cache` to capture decoder-side activations.

In [ ]:
tokenizer = model.tokenizer

# Quick test: encode a source + target pair
source = "The quick brown fox jumps over the lazy dog."
target = "A fox jumps over a dog."

enc_tokens = tokenizer(source, return_tensors="pt")
dec_tokens = tokenizer(target, return_tensors="pt")

print(f"Encoder input shape: {enc_tokens.input_ids.shape}")
print(f"Decoder input shape: {dec_tokens.input_ids.shape}")

# Run with cache and capture decoder.12.hook_mlp_out
with torch.no_grad():
    logits, cache = model.run_with_cache(
        enc_tokens.input_ids,
        decoder_input=dec_tokens.input_ids,
        names_filter=lambda name: name == "decoder.12.hook_mlp_out",
    )

acts = cache["decoder.12.hook_mlp_out"]
print(f"\nDecoder.12 hook_mlp_out shape: {acts.shape}")
print(f"  -> (batch={acts.shape[0]}, seq_len={acts.shape[1]}, d_model={acts.shape[2]})")

## Load Dataset (XSum for summarization)

We use the XSum dataset which provides document-summary pairs,
giving us meaningful encoder (document) and decoder (summary) inputs.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("EdinburghNLP/xsum", split="train", streaming=True)
dataset_iter = iter(dataset)

# Preview a sample
sample = next(dataset_iter)
print(f"Document: {sample['document'][:200]}...")
print(f"Summary:  {sample['summary']}")

## Activation Collection Helper

We collect decoder-side activations by:
1. Tokenizing the source (encoder input) and summary (decoder input) separately
2. Running `model.run_with_cache` with both inputs
3. Extracting activations from the target hook point

In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # SAE architecture
    d_in: int = 1024          # T5-large d_model
    d_sae: int = 16384        # SAE feature count (16x expansion)
    # Hook point
    hook_name: str = "decoder.12.hook_mlp_out"
    # Data
    context_size: int = 128   # max encoder token length
    target_size: int = 64     # max decoder token length
    batch_size: int = 4096    # number of activation tokens per training step
    # Training
    total_steps: int = 10_000
    lr: float = 5e-5
    l1_coefficient: float = 5.0
    l1_warm_up_steps: int = 500
    log_every: int = 100
    n_batches_for_norm_estimate: int = 50

cfg = Config()
print(f"SAE Config: d_in={cfg.d_in}, d_sae={cfg.d_sae}, expansion={cfg.d_sae // cfg.d_in}x")
print(f"Hook: {cfg.hook_name}")
print(f"Training: {cfg.total_steps} steps, batch_size={cfg.batch_size}, l1={cfg.l1_coefficient}")

In [ ]:
@torch.no_grad()
def collect_activations_batch(dataset_iter, model, tokenizer, cfg, target_tokens=4096):
    """Collect decoder activations from a batch of dataset samples.
    
    Returns a flat tensor of shape (n_tokens, d_in) by collecting
    decoder-side activations from multiple samples.
    """
    collected = []
    total_tokens = 0

    while total_tokens < target_tokens:
        try:
            sample = next(dataset_iter)
        except StopIteration:
            break

        source = sample["document"]
        target = sample["summary"]

        if not source.strip() or not target.strip():
            continue

        enc_tokens = tokenizer(
            source, return_tensors="pt", truncation=True,
            max_length=cfg.context_size, padding=False,
        )
        dec_tokens = tokenizer(
            target, return_tensors="pt", truncation=True,
            max_length=cfg.target_size, padding=False,
        )
        n_dec = dec_tokens.input_ids.shape[1]

        _, cache = model.run_with_cache(
            enc_tokens.input_ids,
            decoder_input=dec_tokens.input_ids,
            names_filter=lambda name: name == cfg.hook_name,
        )

        acts = cache[cfg.hook_name].squeeze(0).float()  # (n_dec, d_in)
        collected.append(acts)
        total_tokens += acts.shape[0]

    if not collected:
        return torch.empty(0, cfg.d_in, device=device)

    all_acts = torch.cat(collected, dim=0)
    # Randomly subsample if we collected too many
    if all_acts.shape[0] > target_tokens:
        indices = torch.randperm(all_acts.shape[0])[:target_tokens]
        all_acts = all_acts[indices]
    return all_acts

# Test collection
test_acts = collect_activations_batch(dataset_iter, model, tokenizer, cfg, target_tokens=512)
print(f"Collected activations shape: {test_acts.shape}")
print(f"  -> ({test_acts.shape[0]} tokens, {test_acts.shape[1]} d_model)")

## Create StandardTrainingSAE

We use SAELens's `StandardTrainingSAE` with `StandardTrainingSAEConfig`.
This gives us the same SAE architecture as SAELens uses internally:
- Unit-norm decoder rows with `decoder_init_norm=0.1`
- `encode_with_hidden_pre` -> ReLU -> `decode`
- L1 sparsity loss weighted by decoder row norms

In [ ]:
sae_cfg = StandardTrainingSAEConfig(
    d_in=cfg.d_in,
    d_sae=cfg.d_sae,
    l1_coefficient=cfg.l1_coefficient,
    lp_norm=1.0,
    l1_warm_up_steps=cfg.l1_warm_up_steps,
    apply_b_dec_to_input=False,
    normalize_activations="expected_average_only_in",
    decoder_init_norm=0.1,
    device=device,
    dtype="float32",
)

sae = StandardTrainingSAE(sae_cfg).to(device)

n_params = sum(p.numel() for p in sae.parameters())
print(f"SAE initialized: {n_params:,} parameters")
print(f"  W_enc: {sae.W_enc.shape}")
print(f"  W_dec: {sae.W_dec.shape}")
print(f"  b_enc: {sae.b_enc.shape}")
print(f"  b_dec: {sae.b_dec.shape}")

## Estimate Activation Scaling Factor

When `normalize_activations="expected_average_only_in"`, we need to estimate
a scaling factor so the mean L2 norm of inputs matches `sqrt(d_in)`. This is
the same approach SAELens uses internally.

In [ ]:
import numpy as np

print("Estimating activation scaling factor...")
norms = []
for i in range(cfg.n_batches_for_norm_estimate):
    batch = collect_activations_batch(dataset_iter, model, tokenizer, cfg, target_tokens=4096)
    if batch.shape[0] > 0:
        norms.append(batch.norm(dim=-1).mean().item())
    if (i + 1) % 10 == 0:
        print(f"  Batch {i + 1}/{cfg.n_batches_for_norm_estimate}")

mean_norm = np.mean(norms)
scaling_factor = (cfg.d_in ** 0.5) / mean_norm

print(f"\nMean activation L2 norm: {mean_norm:.4f}")
print(f"Target norm (sqrt(d_in)): {cfg.d_in ** 0.5:.4f}")
print(f"Scaling factor: {scaling_factor:.4f}")

# Store it for use during training
sae.scaling_factor = torch.tensor(scaling_factor, device=device)

## Training Loop

We use SAELens's `training_forward_pass` with `TrainStepInput` for each step.
This gives us the same loss computation as `SAETrainingRunner`:
- MSE reconstruction loss (sum over d_in, mean over batch)
- L1 sparsity loss weighted by decoder row norms
- L1 coefficient linear warm-up

In [ ]:
from tqdm.auto import tqdm

optimizer = torch.optim.Adam(sae.parameters(), lr=cfg.lr, betas=(0.9, 0.999))

# Metrics tracking
metrics = {
    "step": [], "total_loss": [], "mse_loss": [], "l1_loss": [],
    "explained_variance": [], "l0": [], "l1_coeff": [],
}

print(f"\nStarting training for {cfg.total_steps} steps...")
print(f"  Hook: {cfg.hook_name}")
print(f"  d_in={cfg.d_in}, d_sae={cfg.d_sae}")
print(f"  L1 coefficient: {cfg.l1_coefficient} (warm-up: {cfg.l1_warm_up_steps} steps)")
print(f"  Dataset: XSum (streaming)\n")

sae.train()
pbar = tqdm(range(cfg.total_steps), desc="Training SAE on T5 Decoder")

for step in pbar:
    # Collect a batch of activations
    sae_in = collect_activations_batch(
        dataset_iter, model, tokenizer, cfg, target_tokens=cfg.batch_size
    )
    if sae_in.shape[0] == 0:
        continue

    # Apply scaling factor (from normalize_activations="expected_average_only_in")
    sae_in = sae_in * sae.scaling_factor

    # Compute L1 coefficient with linear warm-up
    if step < cfg.l1_warm_up_steps:
        l1_coeff = cfg.l1_coefficient * (step / cfg.l1_warm_up_steps)
    else:
        l1_coeff = cfg.l1_coefficient

    # Forward pass using SAELens's training_forward_pass
    step_output = sae.training_forward_pass(
        step_input=TrainStepInput(
            sae_in=sae_in,
            coefficients={"l1": l1_coeff},
            dead_neuron_mask=None,
            n_training_steps=step,
            is_logging_step=(step % cfg.log_every == 0),
        )
    )

    # Backward + optimize
    optimizer.zero_grad()
    step_output.loss.backward()
    torch.nn.utils.clip_grad_norm_(sae.parameters(), 1.0)
    optimizer.step()

    # Compute metrics
    with torch.no_grad():
        l0 = step_output.feature_acts.bool().float().sum(-1).mean().item()
        per_token_l2 = (step_output.sae_out - sae_in).pow(2).sum(dim=-1)
        total_var = (sae_in - sae_in.mean(0)).pow(2).sum(-1)
        ev = 1 - per_token_l2.mean() / (total_var.mean() + 1e-8)

    # Track metrics
    metrics["step"].append(step)
    metrics["total_loss"].append(step_output.loss.item())
    metrics["mse_loss"].append(step_output.losses["mse_loss"].item())
    metrics["l1_loss"].append(step_output.losses["l1_loss"].item())
    metrics["explained_variance"].append(ev.item())
    metrics["l0"].append(l0)
    metrics["l1_coeff"].append(l1_coeff)

    if step % cfg.log_every == 0:
        pbar.set_postfix({
            "loss": f"{step_output.loss.item():.4f}",
            "mse": f"{step_output.losses['mse_loss'].item():.4f}",
            "ev": f"{ev.item():.4f}",
            "l0": f"{l0:.1f}",
        })

pbar.close()
print("\nTraining complete!")

## Plot Training Metrics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("T5-large Decoder SAE Training Metrics", fontsize=14)

axes[0, 0].plot(metrics["step"], metrics["total_loss"])
axes[0, 0].set_title("Total Loss")
axes[0, 0].set_xlabel("Step")

axes[0, 1].plot(metrics["step"], metrics["mse_loss"])
axes[0, 1].set_title("MSE Loss")
axes[0, 1].set_xlabel("Step")

axes[0, 2].plot(metrics["step"], metrics["l1_loss"])
axes[0, 2].set_title("L1 Loss")
axes[0, 2].set_xlabel("Step")

axes[1, 0].plot(metrics["step"], metrics["explained_variance"])
axes[1, 0].set_title("Explained Variance")
axes[1, 0].set_xlabel("Step")
axes[1, 0].axhline(y=0.8, color="r", linestyle="--", alpha=0.5, label="Target (0.8)")
axes[1, 0].legend()

axes[1, 1].plot(metrics["step"], metrics["l0"])
axes[1, 1].set_title("L0 (Active Features per Token)")
axes[1, 1].set_xlabel("Step")

axes[1, 2].plot(metrics["step"], metrics["l1_coeff"])
axes[1, 2].set_title("L1 Coefficient (with warm-up)")
axes[1, 2].set_xlabel("Step")

plt.tight_layout()
plt.show()

## Evaluation

Collect a larger batch of activations to evaluate the trained SAE.

In [ ]:
sae.eval()

all_feature_acts = []
all_reconstructions = []
all_inputs = []

with torch.no_grad():
    for _ in range(10):
        batch = collect_activations_batch(
            dataset_iter, model, tokenizer, cfg, target_tokens=4096
        )
        if batch.shape[0] == 0:
            continue
        scaled = batch * sae.scaling_factor
        feature_acts, hidden_pre = sae.encode_with_hidden_pre(scaled)
        reconstruction = sae.decode(feature_acts)

        all_feature_acts.append(feature_acts)
        all_reconstructions.append(reconstruction)
        all_inputs.append(scaled)

feature_acts = torch.cat(all_feature_acts, dim=0)
reconstructions = torch.cat(all_reconstructions, dim=0)
inputs = torch.cat(all_inputs, dim=0)

# Explained variance
per_token_mse = (reconstructions - inputs).pow(2).sum(dim=-1)
total_variance = (inputs - inputs.mean(0)).pow(2).sum(dim=-1)
explained_variance = 1 - per_token_mse.mean() / (total_variance.mean() + 1e-8)

# L0 and feature density
active = feature_acts.bool().float()
l0 = active.sum(-1).mean()
feature_density = active.mean(0)
dead_features = (feature_density < 1e-6).sum().item()

print(f"Evaluation Results:")
print(f"  Explained Variance: {explained_variance.item():.4f}")
print(f"  L0 (avg active features): {l0.item():.1f}")
print(f"  Dead features: {dead_features}/{feature_acts.shape[-1]} ({dead_features/feature_acts.shape[-1]*100:.1f}%)")
print(f"  Mean feature density: {feature_density.mean().item():.6f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(feature_density.cpu().numpy(), bins=50, edgecolor="black")
axes[0].set_xlabel("Feature Activation Rate")
axes[0].set_ylabel("Count")
axes[0].set_title("Feature Density Distribution")
axes[0].axvline(x=feature_density.mean().item(), color="r", linestyle="--", label="Mean")
axes[0].legend()

axes[1].hist(l0.cpu().numpy(), bins=50, edgecolor="black")
axes[1].set_xlabel("L0 (Active Features per Token)")
axes[1].set_ylabel("Count")
axes[1].set_title("L0 Distribution")
axes[1].axvline(x=l0.item(), color="r", linestyle="--", label="Mean")
axes[1].legend()

plt.tight_layout()
plt.show()

## Logit Lens: Project Features onto Vocabulary

Project SAE decoder weights onto the T5 embedding matrix to see
what tokens each feature promotes.

In [ ]:
with torch.no_grad():
    # T5 uses separate encoder/decoder embeddings, use decoder embeddings
    embed = model.W_dec  # (d_vocab, d_model)
    projection = sae.W_dec @ embed.T  # (d_sae, d_vocab)

    top_k = 5
    vals, inds = torch.topk(projection, top_k, dim=1)

    # Show 10 random features
    random_indices = torch.randint(0, projection.shape[0], (10,))

    print(f"Top {top_k} tokens promoted by 10 random features:")
    print("-" * 80)
    for idx in random_indices:
        feat = idx.item()
        tokens = [model.to_string(i) for i in inds[feat]]
        scores = [f"{v:.2f}" for v in vals[feat]]
        token_strs = [f"'{t}' ({s})" for t, s in zip(tokens, scores)]
        print(f"Feature {feat:5d}: {', '.join(token_strs)}")

## Save the Trained SAE

In [ ]:
import json
from pathlib import Path
from safetensors.torch import save_file

save_dir = Path("checkpoints/sae_t5_large_decoder")
save_dir.mkdir(parents=True, exist_ok=True)

# Save weights
save_file(
    {
        "W_enc": sae.W_enc.data,
        "W_dec": sae.W_dec.data,
        "b_enc": sae.b_enc.data,
        "b_dec": sae.b_dec.data,
        "scaling_factor": sae.scaling_factor,
    },
    str(save_dir / "sae_weights.safetensors"),
)

# Save config
with open(save_dir / "sae_config.json", "w") as f:
    json.dump({
        "d_in": cfg.d_in,
        "d_sae": cfg.d_sae,
        "hook_name": cfg.hook_name,
        "l1_coefficient": cfg.l1_coefficient,
        "model_name": "google-t5/t5-large",
        "sae_type": "standard_training_sae",
    }, f, indent=2)

# Save metrics
with open(save_dir / "training_metrics.json", "w") as f:
    json.dump(metrics, f)

print(f"Saved to {save_dir}")
print(f"  - sae_weights.safetensors")
print(f"  - sae_config.json")
print(f"  - training_metrics.json")